### Importing Libraries & API Configuration

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

API_KEY = os.environ.get("SECTORS_API_KEY", "")

BASE_URL = "https://api.sectors.app/v2"
HEADERS = {"Authorization": API_KEY}

if not API_KEY:
    raise ValueError("SECTORS_API_KEY not yet set")
print("API Key & Library ready")

API Key & Library ready


### Date Parameters & Cache Folder Setup

In [13]:
END_DATE = datetime.now().strftime("%Y-%m-%d")
START_DATE = (datetime.now() - timedelta(days=180)).strftime("%Y-%m-%d")

CACHE_DIR = "cache/daily"
MANIFEST_PATH = "cache/fetch_manifest.csv"
UNIVERSE_PATH = "cache/universe_tickers.csv"
MARKET_CAP_RAW_PATH = "cache/market_cap_raw.csv"

os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Target period: {START_DATE} to {END_DATE} (180 days)")
print(f"Cache directory: {CACHE_DIR}")

Target period: 2026-03-15 to 2026-09-11 (180 days)
Cache directory: cache/daily


### Micro + Small-Cap Universe Determination Function (Part A)

In [16]:
def fetch_all_market_caps(chunk_size: int = 200) -> list[dict]:
    all_rows = []
    offset = 0
    while True:
        params = {
            "where": "market_cap > 0",
            "order_by": "-market_cap",
            "limit": chunk_size,
            "offset": offset,
            "include_query_values": "true",
        }
        resp = requests.get(f"{BASE_URL}/companies/", headers=HEADERS, params=params, timeout=20)
        resp.raise_for_status()
        payload = resp.json()
        results = payload.get("results", [])
        if not results:
            break
        for r in results:
            qv = r.get("query_values") or {}
            mc = qv.get("market_cap") or r.get("market_cap")
            all_rows.append({
                "symbol": r.get("symbol"),
                "company_name": r.get("company_name"),
                "market_cap": mc,
            })
        pagination = payload.get("pagination", {})
        if not pagination.get("has_next"):
            break
        offset = pagination.get("next_offset", offset + chunk_size)
    return all_rows


def determine_universe() -> tuple[list[str], pd.DataFrame]:
    if os.path.exists(UNIVERSE_PATH):
        print(f"Universe loaded from cache: {UNIVERSE_PATH}")
        df = pd.read_csv(UNIVERSE_PATH)
        return df["symbol"].tolist(), df

    print("Fetching market cap data for all IDX stocks")
    rows = fetch_all_market_caps()
    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(MARKET_CAP_RAW_PATH, index=False)

    caps = raw_df["market_cap"].dropna().astype(float)
    log_caps = np.log10(caps[caps > 0])
    q1, q2, q3 = np.percentile(log_caps, [25, 50, 75])
    small_cap_max = 10 ** q2

    print(f"Total stocks with market cap data: {len(caps)}")
    print(f"Small-cap upper limit (median log-market-cap): Rp {small_cap_max:,.0f}")

    universe_df = raw_df[
        raw_df["market_cap"].notna() & (raw_df["market_cap"] <= small_cap_max)
    ].copy()

    universe_df["symbol"] = universe_df["symbol"].str.replace(".JK", "", regex=False)
    universe_df.to_csv(UNIVERSE_PATH, index=False)

    print(f"Micro+small-cap universe: {len(universe_df)} stocks saved to {UNIVERSE_PATH}")
    return universe_df["symbol"].tolist(), universe_df

### Universe Definition Execution

In [17]:
universe_tickers, universe_df = determine_universe()
universe_df.head()

Universe loaded from cache: cache/universe_tickers.csv


,symbol,company_name,market_cap
0,ADCP,PT Adhi Commuter Properti Tbk,1111111110000
1,HOME,Hotel Mandarine Regency Tbk,1110609739100
2,GDST,Gunawan Dianjaya Steel Tbk,1109100000000
3,PPRE,PT PP Presisi Tbk,1104221268000
4,VERN,PT Verona Indah Pictures Tbk,1096099082090


### Smart Extend & Retry 429 Helper (Part B)

In [18]:
def load_manifest() -> pd.DataFrame:
    if os.path.exists(MANIFEST_PATH):
        return pd.read_csv(MANIFEST_PATH)
    return pd.DataFrame(columns=["symbol", "status", "n_rows", "last_attempt"])

def save_manifest(df: pd.DataFrame) -> None:
    df.to_csv(MANIFEST_PATH, index=False)

def fetch_chunk_with_retry(ticker: str, start: str, end: str, max_retries: int = 5):
    for attempt in range(max_retries):
        try:
            resp = requests.get(
                f"{BASE_URL}/daily/{ticker}/",
                headers=HEADERS,
                params={"start": start, "end": end},
                timeout=20,
            )
            if resp.status_code == 200:
                return pd.DataFrame(resp.json())
            if resp.status_code == 429:
                wait = 2 ** attempt  # 1, 2, 4, 8, 16 detik
                print(f"[429: wait {wait}s] ", end="", flush=True)
                time.sleep(wait)
                continue
            return None
        except requests.exceptions.RequestException:
            time.sleep(1)
            continue
    return None

def extend_ticker_to_full_range(ticker: str, manifest: pd.DataFrame) -> pd.DataFrame:
    cache_path = os.path.join(CACHE_DIR, f"{ticker}.csv")

    if os.path.exists(cache_path):
        existing = pd.read_csv(cache_path)
        existing["date"] = pd.to_datetime(existing["date"])
        earliest_cached = existing["date"].min()
    else:
        existing = pd.DataFrame()
        earliest_cached = pd.to_datetime(END_DATE)

    target_start = pd.to_datetime(START_DATE)

    if earliest_cached <= target_start:
        return existing

    missing_end = (earliest_cached - timedelta(days=1)).strftime("%Y-%m-%d")
    df_new = fetch_chunk_with_retry(ticker, START_DATE, missing_end)

    if df_new is None:
        manifest = manifest[manifest["symbol"] != ticker]
        manifest = pd.concat([manifest, pd.DataFrame([{
            "symbol": ticker, "status": "extend_failed",
            "n_rows": len(existing), "last_attempt": pd.Timestamp.now(),
        }])], ignore_index=True)
        save_manifest(manifest)
        return existing

    if not df_new.empty:
        df_new["date"] = pd.to_datetime(df_new["date"])

    combined = pd.concat([existing, df_new], ignore_index=True)
    combined = combined.drop_duplicates(subset=["date"]).sort_values("date")
    combined.to_csv(cache_path, index=False)

    manifest = manifest[manifest["symbol"] != ticker]
    manifest = pd.concat([manifest, pd.DataFrame([{
        "symbol": ticker, "status": "done",
        "n_rows": len(combined), "last_attempt": pd.Timestamp.now(),
    }])], ignore_index=True)
    save_manifest(manifest)

    return combined

### Implementation of Cache Extension to 180 Days

In [19]:
print(f"Extending cache for {len(universe_tickers)} stocks to the range {START_DATE} to {END_DATE}")

manifest = load_manifest()
all_results = []
n_skipped = 0
n_extended = 0

for i, ticker in enumerate(universe_tickers):
    before = os.path.exists(os.path.join(CACHE_DIR, f"{ticker}.csv"))
    df = extend_ticker_to_full_range(ticker, manifest)
    manifest = load_manifest()

    min_date = df["date"].min() if not df.empty else None
    already_full = before and min_date is not None and pd.to_datetime(min_date) <= pd.to_datetime(START_DATE)

    if already_full:
        n_skipped += 1
        tag = "SKIP"
    else:
        n_extended += 1
        tag = "EXTENDED"
        time.sleep(0.5)

    print(f"  [{i + 1:3d}/{len(universe_tickers)}] {ticker:<8} {tag} ({len(df)} baris)")

    if not df.empty:
        df = df.copy()
        df["symbol"] = ticker
        all_results.append(df)

if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv("cache/daily_all_combined.csv", index=False)
    print(f"\nDone! {combined['symbol'].nunique()} stocks, {len(combined)} rows")
    print(f"Combined date range: {combined['date'].min()} to {combined['date'].max()}")
    print(f"Summary: {n_extended} extended, {n_skipped} skipped")

Extending cache for 481 stocks to the range 2026-03-15 to 2026-09-11
  [  1/481] ADCP     EXTENDED (114 baris)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23080\2242971189.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([existing, df_new], ignore_index=True)


  [  2/481] HOME     EXTENDED (114 baris)
  [  3/481] GDST     EXTENDED (114 baris)
  [  4/481] PPRE     EXTENDED (114 baris)
  [  5/481] VERN     EXTENDED (114 baris)
  [  6/481] BBLD     EXTENDED (114 baris)
  [  7/481] BBRM     EXTENDED (114 baris)
  [  8/481] AISA     EXTENDED (114 baris)
  [  9/481] ADMG     EXTENDED (114 baris)
  [ 10/481] JIHD     EXTENDED (114 baris)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23080\2242971189.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([existing, df_new], ignore_index=True)


  [ 11/481] TAMU     EXTENDED (114 baris)
  [ 12/481] ARTA     EXTENDED (114 baris)
  [ 13/481] CITY     EXTENDED (114 baris)
  [ 14/481] PTPW     EXTENDED (114 baris)
  [ 15/481] TCID     EXTENDED (114 baris)


...

  [479/481] DEAL     EXTENDED (114 baris)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23080\2242971189.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([existing, df_new], ignore_index=True)


  [480/481] MKNT     EXTENDED (114 baris)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23080\2242971189.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([existing, df_new], ignore_index=True)


  [481/481] SBAT     EXTENDED (114 baris)

Done! 481 stocks, 54150 rows
Combined date range: 2026-03-16 00:00:00 to 2026-09-10 00:00:00
Summary: 481 extended, 0 skipped


### Validation of 180-Day Final Result Data

In [20]:
df_daily = pd.read_csv("cache/daily_all_combined.csv")

print(f"Total Unique Stocks : {df_daily['symbol'].nunique()}")
print(f"Total Data Rows : {len(df_daily):,}")
print(f"Date Range : {df_daily['date'].min()} to {df_daily['date'].max()}")
print(f"Average rows per stock: {len(df_daily) / df_daily['symbol'].nunique():.0f} trading days")

df_daily.head()

Total Unique Stocks : 481
Total Data Rows : 54,150
Date Range : 2026-03-16 to 2026-09-10
Average rows per stock: 113 trading days


,symbol,date,close,open,high,low,volume,market_cap
0,ADCP,2026-03-16,50,50.0,51,50,1280400,1111111110000
1,ADCP,2026-03-17,50,50.0,51,50,2432700,1111111110000
2,ADCP,2026-03-25,50,50.0,51,50,1218200,1111111110000
3,ADCP,2026-03-26,50,50.0,51,50,3241300,1111111110000
4,ADCP,2026-03-27,50,50.0,51,50,858500,1111111110000
